In [17]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [18]:
df = pd.read_csv(
    "../data/processed/bitcoin_features.csv"
)

df.head()

,Date,Open,High,Low,Close,Adj Close,Volume,Daily_return,Volatility,MA7,...,Price_vs_MA30,MA7_vs_MA30_ratio,Close_vs_MA30_ratio,Delta,Gain,Loss,Avg_gain,Avg_loss,RS,RSI
0,2014-12-15,351.360992,351.815002,344.933990,345.345001,345.345001,17264200,-0.017879,0.021259,349.426426,...,-5.988493,0.951226,0.940115,-6.286987,0.000000,6.286987,1.433071,3.854499,0.371792,27.102635
1,2014-12-16,345.673004,345.859009,327.062012,327.062012,327.062012,30864900,-0.052941,0.022171,345.832572,...,-10.471479,0.946667,0.895285,-18.282990,0.000000,18.282990,1.285213,5.160427,0.249052,19.939262
2,2014-12-17,326.855011,333.954010,315.152008,319.776001,319.776001,37567900,-0.022277,0.022366,342.034145,...,-11.922384,0.942083,0.880776,-7.286011,0.000000,7.286011,1.285213,5.230499,0.245715,19.724827
3,2014-12-18,319.785004,323.709015,304.231995,311.395996,311.395996,39173000,-0.026206,0.022178,336.447000,...,-13.725155,0.932154,0.862748,-8.380005,0.000000,8.380005,1.285213,5.442928,0.236125,19.102052
4,2014-12-19,311.178986,318.532990,306.769012,317.842987,317.842987,23823100,0.020704,0.022410,331.489999,...,-11.425975,0.923771,0.885740,6.446991,6.446991,-0.000000,1.227855,5.442928,0.225587,18.406462


# Predicting Bitcoin Direction

Objective:

Investigate whether technical indicators can predict future Bitcoin price movement.

In [19]:
df["Target"]=(df["Close"].shift(-1)>df["Close"]).astype(int)
df["target_7D"]=(df["Close"].shift(-7)>df["Close"]).astype(int)

In [20]:
df["Target"].tail(10)

4181    0
4182    0
4183    1
4184    0
4185    0
4186    0
4187    0
4188    0
4189    0
4190    0
Name: Target, dtype: int64

In [21]:
df["Target"].value_counts()

Target
1    2199
0    1992
Name: count, dtype: int64

In [22]:
df_ml=df.dropna().copy()
df_ml.shape

(4191, 24)

In [23]:
features=["Daily_return","Volatility","RSI","Close_vs_MA30_ratio","MA7_vs_MA30_ratio"]

In [24]:
X=df_ml[features]
Y=df_ml["Target"]
Z=df_ml["target_7D"]
print(X.head())
print()
print(Y.head())
print()
print(Z.head())

   Daily_return  Volatility        RSI  Close_vs_MA30_ratio  MA7_vs_MA30_ratio
0     -0.017879    0.021259  27.102635             0.940115           0.951226
1     -0.052941    0.022171  19.939262             0.895285           0.946667
2     -0.022277    0.022366  19.724827             0.880776           0.942083
3     -0.026206    0.022178  19.102052             0.862748           0.932154
4      0.020704    0.022410  18.406462             0.885740           0.923771

0    0
1    0
2    0
3    1
4    1
Name: Target, dtype: int64

0    0
1    1
2    1
3    1
4    1
Name: target_7D, dtype: int64


In [25]:
split=int(len(df_ml)*0.8)
X_train=X[:split]
X_test=X[split:]
Y_train=Y[:split]
Y_test=Y[split:]
Z_train=Z[:split]
Z_test=Z[split:]

In [26]:
print(X_train.shape)
print(X_test.shape)

(3352, 5)
(839, 5)


In [27]:
model1=LogisticRegression(max_iter=1000)
model1.fit(X_train,Y_train)
model2=LogisticRegression(max_iter=1000)
model2.fit(X_train,Z_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [28]:
pred1=model1.predict(X_test)
pred2=model2.predict(X_test)

In [29]:
accuracy1=accuracy_score(Y_test,pred1)
print(accuracy1)
accuracy2=accuracy_score(Z_test,pred2)
print(accuracy2)

0.4922526817640048
0.5113230035756854


In [30]:
print(classification_report(Y_test,pred1))
print(classification_report(Z_test,pred2))

              precision    recall  f1-score   support

           0       0.47      0.08      0.14       421
           1       0.49      0.91      0.64       418

    accuracy                           0.49       839
   macro avg       0.48      0.49      0.39       839
weighted avg       0.48      0.49      0.39       839

              precision    recall  f1-score   support

           0       0.46      0.08      0.14       405
           1       0.52      0.91      0.66       434

    accuracy                           0.51       839
   macro avg       0.49      0.50      0.40       839
weighted avg       0.49      0.51      0.41       839



In [31]:
for feature, coef in zip(features,model1.coef_[0]):
  print(feature, coef)

Daily_return -1.5854249912916363
Volatility -0.1781744153830125
RSI 0.0068801275039943755
Close_vs_MA30_ratio -0.8277268587564811
MA7_vs_MA30_ratio 0.558385578485958


In [32]:
corr = df_ml[[
    "Daily_return",
    "Volatility",
    "RSI",
    "Close_vs_MA30_ratio",
    "MA7_vs_MA30_ratio",
    "target_7D"
]].corr()

print(corr["target_7D"].sort_values(ascending=False))

target_7D              1.000000
RSI                    0.060660
MA7_vs_MA30_ratio      0.042103
Close_vs_MA30_ratio    0.036254
Volatility             0.002403
Daily_return          -0.013400
Name: target_7D, dtype: float64


# Random Forest Classification

Unlike Logistic Regression, Random Forest can learn non-linear relationships and interactions between features.

Objective:
Determine whether a more powerful model can extract predictive signals from technical indicators.

In [34]:
rf=RandomForestClassifier(n_estimators=200,max_depth=5,random_state=42)
rf.fit(X_train,Y_train)
pred_rf=rf.predict(X_test)

In [35]:
rf_acc=accuracy_score(Y_test,pred_rf)
print(rf_acc)

0.5208581644815257


In [36]:
print(classification_report(Y_test,pred_rf))

              precision    recall  f1-score   support

           0       0.53      0.38      0.44       421
           1       0.51      0.66      0.58       418

    accuracy                           0.52       839
   macro avg       0.52      0.52      0.51       839
weighted avg       0.52      0.52      0.51       839

